# Chapter 3 — Giving Agents Tools

In Chapter 1 a tool was a function in a dictionary. That works, and it does not scale:
every tool needs a hand-written schema, nothing is shared between agents, and a tool
added by one team is invisible to every other.

There are three mechanisms, and they differ in who owns the schema, when the schema is
known, and how much reuse you get — not in what the agent can do.

**Covered in this lab:** §3.2 function calling · §3.3 structured output *and
validation before execution* · §3.5.2 generating tools from OpenAPI · §3.5.3 MCP
runtime discovery (the real SDK) · §3.5.4 choosing a strategy.


## Setup

Every lab in this book installs from **one** `requirements.txt` in the companion
repository. No notebook pins its own versions: change a dependency there and it
changes everywhere, including CI. That is how the labs mirror a production
service rather than a pile of scratch files.

The clone below fails loudly on purpose. A setup step that swallows its own
error surfaces later as a confusing `ModuleNotFoundError`, and you waste an hour
looking in the wrong place.


In [ ]:
REPO_URL = "https://github.com/<your-org>/<your-repo>.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


In [ ]:
!pip -q install -r requirements.txt


Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon.


In [ ]:
!python tools/check_env.py


### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [ ]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


## §3.2 — Function calling

You hand-write the schema: name, description, and a JSON Schema for the arguments.
The model reads it to decide what to call and with what.

Count the characters. Now imagine forty tools. That cost is the motivation for the
next two mechanisms.


In [ ]:
import json, sys
sys.path.insert(0, "ch03")

from function_calling.tools_fc import (IP_REPUTATION_SCHEMA, SCHEMAS as FC_SCHEMAS,
                                       REGISTRY as FC_REGISTRY, dispatch as fc_dispatch)

print("function calling — hand-written schema:")
print(f'  {len(json.dumps(IP_REPUTATION_SCHEMA))} chars of JSON, written by a human')
print(f'  tools exposed: {[s["function"]["name"] for s in FC_SCHEMAS]}')
print()

verdict = json.loads(fc_dispatch("ip_reputation", {"ip": "203.0.113.42"}))
print("  result:", verdict["verdict"], f'(score {verdict["score"]})')


## §3.3 — Structured output, and validating it before execution

A model emits a tool call as *text*. Text can be wrong in three ways that matter, and
all three arrive looking identical: a tool that does not exist, arguments that do not
match, or output that is not valid JSON at all.

Validation turns each of those into **data** instead of an exception. The rule is
**fail closed**: a rejected call is something you can log, count, and alert on. A crash
is none of those things.

This is also the seam Chapter 11 builds on — the function that rejects a malformed
call is the natural place to reject an *unauthorized* one.


In [ ]:
from validation import validate_tool_call, guarded_dispatch

CASES = [
    ("valid call    ", json.dumps({"name": "ip_reputation",
                                   "arguments": {"ip": "203.0.113.42"}})),
    ("unknown tool  ", json.dumps({"name": "delete_all_logs", "arguments": {}})),
    ("bad arguments ", json.dumps({"name": "ip_reputation", "arguments": {"wrong": 1}})),
    ("malformed json", "{not json at all"),
]

for label, raw in CASES:
    outcome = guarded_dispatch(raw, FC_REGISTRY, FC_SCHEMAS)
    result = (outcome["result"][:34] + "...") if outcome["result"] else "-"
    print(f'{label}  ok={str(outcome["ok"]):5} reason={str(outcome["reason"]):15} {result}')

print()
print("Nothing raised. Every rejection is a value the agent can act on.")


## §3.5.2 — Generating the tool registry from an OpenAPI spec

Most companies already have an API specification for their internal services. If you
have one, you do not write tool schemas at all — you generate them.

The cell below adds an endpoint to the spec and re-runs the converter. You write no
schema; the agent gains a tool.


In [ ]:
import copy
from openapi.tools_openapi import SOC_OPENAPI, openapi_to_schemas

before = openapi_to_schemas(SOC_OPENAPI)
print(f'schemas before: {len(before)}  {[s["function"]["name"] for s in before]}')

extended = copy.deepcopy(SOC_OPENAPI)
extended["paths"]["/identity/user"] = {
    "post": {"operationId": "user_context",
             "summary": "Fetch an account's role, department, and privilege level.",
             "requestBody": {"content": {"application/json": {"schema": {
                 "type": "object",
                 "properties": {"user": {"type": "string"}},
                 "required": ["user"]}}}}}
}

after = openapi_to_schemas(extended)
print(f'schemas after:  {len(after)}  {[s["function"]["name"] for s in after]}')
print()
print("hand-written JSON schema: 0 characters")


## §3.5.3 — MCP: runtime discovery

The Model Context Protocol inverts the relationship. Instead of the client knowing
what tools exist, a **server advertises** them and any client discovers them at
runtime.

That is a genuinely different property. With function calling and OpenAPI the tool
list is compiled into the client. With MCP, a tool added to the server this morning is
available to every agent this afternoon — no client change, no redeploy.

This uses the real `mcp` SDK over its in-memory transport, so it runs offline.


In [ ]:
import asyncio
from mcp.shared.memory import create_connected_server_and_client_session
from mcp_track.tools_mcp import build_soc_server


async def discover_and_call():
    server = build_soc_server()
    async with create_connected_server_and_client_session(server) as client:
        await client.initialize()
        listed = await client.list_tools()                      # DISCOVERY
        names = [t.name for t in listed.tools]
        called = await client.call_tool("ip_reputation", {"ip": "203.0.113.42"})
        return names, json.loads(called.content[0].text)


names, result = asyncio.run(discover_and_call())

print("tools discovered at runtime:", names)
print("nothing about these was hard-coded on the client")
print()
print("called ip_reputation ->", result["verdict"])


## §3.5.4 — Choosing a strategy

Three mechanisms, one verdict. Check the thing that actually matters, then choose on
the things that differ.


In [ ]:
fc_verdict = json.loads(fc_dispatch("ip_reputation", {"ip": "203.0.113.42"}))["verdict"]
oa_verdict = json.loads(
    __import__("openapi.tools_openapi", fromlist=["dispatch"]).dispatch(
        "ip_reputation", {"ip": "203.0.113.42"}))["verdict"]
mcp_verdict = result["verdict"]

print("verdict by mechanism:")
print("  function calling:", fc_verdict)
print("  openapi:         ", oa_verdict)
print("  mcp:             ", mcp_verdict)
assert fc_verdict == oa_verdict == mcp_verdict
print("  -> identical. the mechanism does not change the answer.")
print()

TRADEOFFS = [
    ("Function calling", "high (hand-write each)", "no", "no"),
    ("OpenAPI", "low (one spec -> N tools)", "via shared spec", "no"),
    ("MCP", "low (server declares)", "yes (any client)", "yes"),
]

print(f'  {"mechanism":18} {"effort per tool":26} {"cross-agent reuse":18} runtime discovery')
for mechanism, effort, reuse, discovery in TRADEOFFS:
    print(f'  {mechanism:18} {effort:26} {reuse:18} {discovery}')
print()
print("Rule of thumb: one tool in one process -> function calling.")
print("An API you already have a spec for -> OpenAPI.")
print("Tools shared across agents or teams -> MCP earns its complexity.")


### One warning before you ship any of this

Look at what a tool definition actually sends the model: a name, and a **description
in prose**. That description goes into the model's context — and in the MCP case it
comes from a server you may not own.

The callable can be perfectly correct while the prose is the attack. That is tool
poisoning, and Chapter 11 builds the defense: screen the description, fingerprint what
you approved, and detect the rug pull when a server rewrites it later.


---

## What you built

The same tool wired three ways to the same verdict, a validator that fails closed, and
runtime discovery over the real MCP protocol.

- **The mechanism does not change the answer.** It changes schema effort, reuse, and
  whether tools can be discovered rather than compiled in.
- **Validate before you execute.** A rejected call is data; a crash is not.
- **A tool description is an instruction channel.** A registry you do not govern is a
  supply chain you do not control.

**Next:** Chapter 4 gives Aegis a conversation — interviewing an employee about a
suspicious email and producing a structured incident record.
